In [ ]:
%run ./tmdbapi.ipynb

In [ ]:
import requests
url='https://api.themoviedb.org/3/genre/movie/list?api_key='+API_key
response=requests.get(url)
response.json()


In [ ]:
def get_data(pageno,year,genre):
  url='https://api.themoviedb.org/3/discover/movie?api_key='+API_key+'&primary_release_year.gte='+str(year)+'&sort_by=popularity.desc&with_genres='+str(genre)+'&page='+str(pageno)
  response=requests.get(url)
  if response.status_code==200:
    data=response.json()
    return data['results']
  else:
    return 'error'

In [ ]:
import pandas as pd
df=pd.DataFrame(columns=['Movie ID','Movie Name'])

In [ ]:
import time
pageno=1
while(len(df)<300):

  query_results=get_data(pageno,2000,35)
  if query_results !='error':
    for movie in query_results:
      df.loc[len(df)]=[movie['id'],movie['title']]

  pageno+=1
  time.sleep(0.25)



In [ ]:
len(df['Movie ID'].unique())

In [ ]:
def get_similar_movie(movieid):
  url='https://api.themoviedb.org/3/movie/'+str(movieid)+'/similar?api_key='+API_key
  response=requests.get(url)
  if response.status_code==200:
    return response.json()['results']
  else:
    return 'error'




In [ ]:
get_similar_movie(1204680)

In [ ]:
df1=pd.DataFrame(columns=['Movie ID','Similar Movie ID'])
for movieid in df['Movie ID']:
  query_results=get_similar_movie(movieid)
  if query_results!='error':
    for i in range(min(5,len(query_results))):
      df1.loc[len(df1)]=[movieid,query_results[i]['id']]

In [ ]:
df1.head(10)

In [ ]:
def get_imdb_id(movieid):
  url='https://api.themoviedb.org/3/movie/'+str(movieid)+'/external_ids?api_key='+str(API_key)
  response=requests.get(url)
  if response.status_code==200:
    return response.json()
  else:
    return 'error'

In [ ]:
get_imdb_id(1204680)

In [ ]:
df['IMDB_ID']=None

In [ ]:
for i,movieid in enumerate(df['Movie ID']):
  query_results=get_imdb_id(movieid)
  if query_results!='error':
    df.loc[i,'IMDB_ID']=query_results['imdb_id']

In [ ]:
IMDB_data=pd.read_csv('/content/title.ratings.tsv',sep='\t')

In [ ]:
df=df.merge(IMDB_data,how='left',left_on='IMDB_ID',right_on='tconst')

In [ ]:
df['IMDB_URL']='https://www.imdb.com/title/'+df['IMDB_ID'].fillna(' ')+'/'

In [ ]:
df

In [ ]:
df.rename(columns={'averageRating':'IMDB_Ratings','numVotes':'IMDB Votes'},inplace=True)
df.drop('tconst',axis=1)

In [ ]:
df